# BananaQ — reproducible MobileNetV2 training and Android export
Run in a fresh Google Colab GPU runtime using **Runtime → Run all**. Put `BananaQDataset.zip` in My Drive first. No package installation is required in the usual Colab runtime.

Input: float32 RGB `[1,224,224,3]`, nearest-neighbor stretch, pixels divided by 255. Output order: **Black Sigatoka, Cordana Leaf Spot, Healthy, Panama Disease**. MobileNetV2 normalization is embedded; do not normalize twice.

Use either unsplit class folders or a root containing `train`, `val` (or `validation`), and `test`, each with four class folders. Existing partitions are preserved. Never merge existing test data into training. A lone `train` directory requires an explicit root override and produces only an internal holdout, not a recovered original test set.

Exact decoded-pixel duplicates are detected. Related images, crops, recompressed copies and pre-augmented variants still require source IDs: optionally provide `GROUP_CSV` with `path,group_id` (paths relative to DATASET_ROOT; IDs identify the same original leaf/plant/session across classes). Without source metadata, image-level splits cannot establish source independence. Use separate field photos to assess real deployment performance.


In [ ]:
from google.colab import drive
from pathlib import Path
import csv
import hashlib
import importlib.metadata
import json
import platform
import random
import re
import tempfile
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
LABELS = ['Black Sigatoka', 'Cordana Leaf Spot', 'Healthy', 'Panama Disease']
CLASS_FOLDERS = ['black_sigatoka', 'cordana', 'healthy', 'panama']  # same order as LABELS
DATASET_ROOT_OVERRIDE = None  # relative to extraction directory, e.g. 'BananaQDataset'
GROUP_CSV = None  # optional absolute Drive path to source-group metadata
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
drive.mount('/content/drive')
ZIP_PATH = Path('/content/drive/MyDrive/BananaQDataset.zip')
# Unique run folders prevent stale model checkpoints/results from earlier runs.
RUN_ID = __import__('datetime').datetime.now(__import__('datetime').timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = Path('/content/drive/MyDrive/BananaQ_MobileNetV2') / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
assert ZIP_PATH.is_file(), f'Missing dataset: {ZIP_PATH}'
environment = {'python': platform.python_version(), 'packages': {
    name: importlib.metadata.version(name) for name in ['tensorflow', 'keras', 'numpy', 'pandas', 'scikit-learn', 'Pillow']}}
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2))
print(environment)
print('GPUs:', tf.config.list_physical_devices('GPU'))
# Reproduction also requires the same ZIP, split manifest, software and hardware.


## Dataset inspection and leakage checks
Corrupt images and conflicting duplicate labels stop the run. Identical decoded RGB images within a partition are counted once; duplicates across supplied partitions stop the run. Both original and retained counts are shown. Source grouping takes precedence over exact-image grouping when metadata is provided.


In [ ]:
EXTRACT_DIR = Path(tempfile.mkdtemp(prefix='BananaQDataset_', dir='/content'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in archive.infolist():
        if not (EXTRACT_DIR / member.filename).resolve().is_relative_to(EXTRACT_DIR.resolve()):
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(EXTRACT_DIR)

def normalized(name):
    return re.sub(r'[^a-z0-9]', '', name.lower())

aliases = [set(map(normalized, names)) for names in [
    ['black_sigatoka', 'Black Sigatoka'], ['cordana', 'Cordana Leaf Spot'],
    ['healthy'], ['panama', 'Panama Disease']]]
def class_dirs(root):
    children = [p for p in root.iterdir() if p.is_dir() and not p.name.startswith(('.', '__'))]
    matches = [[p for p in children if normalized(p.name) in names] for names in aliases]
    if len(children) == 4 and all(len(m) == 1 for m in matches):
        return [m[0] for m in matches]
    return None

def partitions(root):
    children = {p.name.lower(): p for p in root.iterdir() if p.is_dir()}
    val_names = [k for k in ['val', 'valid', 'validation'] if k in children]
    if 'train' in children and 'test' in children and len(val_names) == 1:
        result = dict(train=children['train'], validation=children[val_names[0]], test=children['test'])
        if all(class_dirs(p) for p in result.values()):
            return result
    return None

folders = [EXTRACT_DIR, *sorted(p for p in EXTRACT_DIR.rglob('*') if p.is_dir() and '__MACOSX' not in p.parts)]
if DATASET_ROOT_OVERRIDE is not None:
    DATASET_ROOT = EXTRACT_DIR / DATASET_ROOT_OVERRIDE
else:
    split_roots = [p for p in folders if partitions(p)]
    roots = split_roots or [p for p in folders if class_dirs(p) and p.name.lower() not in {'train','val','valid','validation','test'}]
    if len(roots) != 1:
        print([str(p.relative_to(EXTRACT_DIR)) for p in folders][:100])
        raise ValueError('Choose exactly one DATASET_ROOT_OVERRIDE; do not silently select a train folder.')
    DATASET_ROOT = roots[0]
supplied_splits = partitions(DATASET_ROOT)
if not supplied_splits and not class_dirs(DATASET_ROOT):
    raise ValueError('Expected four class folders or complete train/validation/test partitions.')
source_groups = {}
if GROUP_CSV is not None:
    with open(GROUP_CSV, newline='') as stream:
        for row in csv.DictReader(stream):
            key = Path(row['path']).as_posix()
            if key in source_groups or not row['group_id'].strip():
                raise ValueError(f'Duplicate path or empty group ID: {key}')
            source_groups[key] = row['group_id'].strip()
rows = []
for split, root in (supplied_splits or {'unsplit': DATASET_ROOT}).items():
    for label, folder in enumerate(class_dirs(root)):
        for path in sorted(folder.rglob('*')):
            if path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                continue
            try:
                with Image.open(path) as im:
                    rgb = im.convert('RGB')
                    digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
            except Exception as error:
                raise ValueError(f'Unreadable image: {path}') from error
            relative = path.relative_to(DATASET_ROOT).as_posix()
            if GROUP_CSV is not None and relative not in source_groups:
                raise ValueError(f'Missing source group: {relative}')
            rows.append(dict(path=str(path), relative_path=relative, label=label,
                label_name=LABELS[label], split=split, pixel_hash=digest,
                group=source_groups.get(relative, digest)))
frame = pd.DataFrame(rows)
if frame.empty:
    raise ValueError('No supported images found.')
print('Original counts:')
print(pd.crosstab(frame.label_name, frame.split).reindex(LABELS, fill_value=0))
for digest, copies in frame.groupby('pixel_hash'):
    if copies.label.nunique() > 1 or copies.split.nunique() > 1 or copies.group.nunique() > 1:
        raise ValueError(f'Duplicate image crosses labels, splits or source IDs: {copies.relative_path.tolist()}')
print('Removed identical RGB copies:', int(frame.duplicated('pixel_hash').sum()))
frame = frame.drop_duplicates('pixel_hash').reset_index(drop=True)
if GROUP_CSV is None:
    print('SOURCE INDEPENDENCE UNVERIFIED: supply source IDs for related photos or augmented variants.')
print('Dataset:', DATASET_ROOT)


## Train / validation / test split and class balance
Unsplit images use stratified 70/15/15 partitions with seed 42. Source metadata uses stratified group folds (approximately these fractions; exact sizes depend on groups). Supplied partitions are preserved. Insufficient class coverage stops the run.

No oversampling or synthetic duplication: class weights use only retained training counts, and activate when the largest training class has at least 1.5 times the smallest. This transparent heuristic avoids discarding data; inspect counts and label quality before trusting extreme weights. Weighted training loss and unweighted validation loss have different objectives.


In [ ]:
if supplied_splits:
    for group, members in frame.groupby('group'):
        if members.split.nunique() > 1:
            raise ValueError(f'Source group crosses supplied splits: {group}')
else:
    if GROUP_CSV is None:
        development, test = train_test_split(frame.index, test_size=TEST_FRACTION,
            stratify=frame.label, random_state=SEED)
        train, validation = train_test_split(development,
            test_size=VALIDATION_FRACTION / (1 - TEST_FRACTION),
            stratify=frame.loc[development, 'label'], random_state=SEED)
    else:
        def group_holdout(data, fraction):
            folds = StratifiedGroupKFold(n_splits=round(1 / fraction), shuffle=True, random_state=SEED)
            keep, held = next(folds.split(data, data.label, data.group))
            return data.index[keep], data.index[held]
        development, test = group_holdout(frame, TEST_FRACTION)
        train, validation = group_holdout(frame.loc[development], VALIDATION_FRACTION / (1 - TEST_FRACTION))
    for name, indices in [('train', train), ('validation', validation), ('test', test)]:
        frame.loc[indices, 'split'] = name
for name in ['train', 'validation', 'test']:
    if set(frame.loc[frame.split == name, 'label']) != set(range(4)):
        raise ValueError(f'{name} lacks a class; add independent data/groups. Do not search seeds for better scores.')
assert frame.groupby('group').split.nunique().max() == 1
assert frame.groupby('pixel_hash').split.nunique().max() == 1
counts = pd.crosstab(frame.label_name, frame.split).reindex(LABELS)
print('Final unique-image distribution:')
print(counts)
counts.plot.bar(rot=25, ylabel='Unique images', title='Final class distribution')
plt.tight_layout()
plt.show()
frame.drop(columns='path').to_csv(OUTPUT_DIR / 'split_manifest.csv', index=False)
training_counts = counts['train'].to_numpy()
class_weight = ({i: float(training_counts.sum() / (4 * count)) for i, count in enumerate(training_counts)}
    if training_counts.max() / training_counts.min() >= 1.5 else None)
print('Class weights:', class_weight or 'None (limited imbalance)')


## Preprocessing and training-only augmentation
Pillow RGB loading and nearest-neighbor stretch follow the existing notebook/app contract. Pixels are divided by 255 once. Mild rotations, translations, zoom, horizontal flips and brightness variation affect training only. These loaders are retained for continuity; no augmented images are written into the dataset. Preview examples to check that lesions remain visible.


In [ ]:
train_preprocessor = ImageDataGenerator(rescale=1/255., rotation_range=10,
    width_shift_range=0.05, height_shift_range=0.05, zoom_range=0.1,
    horizontal_flip=True, brightness_range=(0.95, 1.05), fill_mode='nearest')
evaluation_preprocessor = ImageDataGenerator(rescale=1/255.)
loader_options = dict(x_col='path', y_col='label_name', classes=LABELS,
    target_size=IMAGE_SIZE, color_mode='rgb', batch_size=BATCH_SIZE,
    class_mode='categorical', interpolation='nearest', seed=SEED)
def make_loader(split, training=False):
    return (train_preprocessor if training else evaluation_preprocessor).flow_from_dataframe(
        frame.loc[frame.split == split].copy(), shuffle=training, **loader_options)
training_data = make_loader('train', training=True)
training_evaluation_data = make_loader('train')
validation_data = make_loader('validation')
test_data = make_loader('test')
assert training_data.class_indices == dict(zip(LABELS, range(4)))
preview, preview_labels = training_data[0]
fig, axes = plt.subplots(1, min(4, len(preview)), figsize=(12, 3), squeeze=False)
for axis, image, label in zip(axes.flat, preview, preview_labels):
    axis.imshow(np.clip(image, 0, 1)); axis.set_title(LABELS[label.argmax()]); axis.axis('off')
plt.tight_layout(); plt.show()
# Recreate the training iterator so previewing does not advance augmentation state.
training_data = make_loader('train', training=True)


## Model, callbacks and training
Keep the original frozen ImageNet MobileNetV2 + pooling + Dense(128) + Dropout(0.4) architecture. Ten epochs remain the upper bound; validation loss selects the checkpoint and drives early stopping / learning-rate reduction. No test-based tuning. Fine-tuning is a future validation-only experiment if the frozen features underfit, not an automatic extra phase.


In [ ]:
base_model = MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3),
)
base_model.trainable = False
inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name='rgb_0_to_1')
x = keras.layers.Rescaling(scale=2.0, offset=-1.0, name='mobilenet_normalization')(inputs)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.4)(x)
outputs = keras.layers.Dense(len(LABELS), activation='softmax', name='disease_probabilities')(x)
model = keras.Model(inputs, outputs, name='BananaQ_MobileNetV2')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy', metrics=['accuracy'],
)
model.summary()
BEST_MODEL_PATH = OUTPUT_DIR / 'BananaQ_MobileNetV2.keras'
history = model.fit(
    training_data, validation_data=validation_data, epochs=EPOCHS, class_weight=class_weight,
    callbacks=[
        keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor='val_loss', save_best_only=True),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    ],
)
model = keras.models.load_model(BEST_MODEL_PATH)


## Learning curves and independent evaluation
Test results are measured only after selecting the validation checkpoint. Do not repeatedly tune against this holdout; use a new test cohort after changes driven by these results. Training evaluation below is unaugmented and unweighted, allowing comparison with validation/test loss. Small class supports make metrics uncertain.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric in zip(axes, ['accuracy', 'loss']):
    axis.plot(range(1, len(history.history[metric])+1), history.history[metric], label='Training')
    axis.plot(range(1, len(history.history[metric])+1), history.history['val_'+metric], label='Validation')
    axis.set(title=metric.title(), xlabel='Epoch'); axis.legend()
plt.tight_layout(); plt.show()
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
results = {}
scores_by_split = {}
for name, loader in [('train', training_evaluation_data), ('validation', validation_data), ('test', test_data)]:
    loader.reset()
    loss, accuracy = model.evaluate(loader, verbose=0)
    loader.reset()
    scores = model.predict(loader, verbose=0)
    scores_by_split[name] = scores
    report = classification_report(loader.classes, scores.argmax(1), labels=list(range(4)),
        target_names=LABELS, zero_division=0, output_dict=True)
    results[name] = dict(loss=float(loss), accuracy=float(accuracy), classification_report=report)
    print(f'{name}: loss={loss:.4f}, accuracy={accuracy:.2%}')
print(classification_report(test_data.classes, scores_by_split['test'].argmax(1),
    labels=list(range(4)), target_names=LABELS, zero_division=0))
per_class = pd.DataFrame(results['test']['classification_report']).T.loc[LABELS]
print('Lowest test F1:', per_class['f1-score'].idxmin())
print(per_class)
ConfusionMatrixDisplay(confusion_matrix(test_data.classes, scores_by_split['test'].argmax(1),
    labels=list(range(4))), display_labels=LABELS).plot(xticks_rotation=45, cmap='Greens')
plt.tight_layout(); plt.show()


## Confidence reliability and high-confidence mistakes
Measure negative log likelihood (NLL), multiclass Brier score (sum over classes) and 10-bin expected calibration error (ECE). ECE depends on sample size/binning and is not proof of reliability. Inspect wrong predictions at ≥95% confidence for label errors, background shortcuts or confusing lesions. A four-class softmax cannot reliably reject unrelated images; evaluate separate non-leaf/out-of-domain photos before deployment.

Raw probabilities are preserved. If diagnostics justify temperature scaling, fit its single parameter on a separate calibration subset of development data, verify with untouched test data, and embed the same transformation for Android export. Do not pick a temperature on test results or merely reduce displayed confidence.


In [ ]:
def calibration_metrics(y, probabilities):
    confidence = probabilities.max(1)
    correct = probabilities.argmax(1) == y
    rows = []
    for index in range(10):
        mask = (confidence >= index/10) & ((confidence < (index+1)/10) if index < 9 else (confidence <= 1))
        if mask.any():
            rows.append(dict(count=int(mask.sum()), confidence=float(confidence[mask].mean()),
                accuracy=float(correct[mask].mean())))
    ece = sum(row['count']/len(y) * abs(row['accuracy']-row['confidence']) for row in rows)
    return dict(nll=float(-np.log(np.clip(probabilities[np.arange(len(y)), y], 1e-7, 1)).mean()),
        brier=float(np.square(probabilities-np.eye(4)[y]).sum(1).mean()), ece=float(ece), bins=rows)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], '--', color='gray')
for name, loader in [('validation', validation_data), ('test', test_data)]:
    probabilities = scores_by_split[name]
    reliability = calibration_metrics(loader.classes, probabilities)
    results[name]['calibration'] = reliability
    print(name, {k:v for k,v in reliability.items() if k != 'bins'})
    ax.plot([b['confidence'] for b in reliability['bins']], [b['accuracy'] for b in reliability['bins']], 'o-', label=name)
    subset = frame.loc[frame.split == name].copy()
    subset['predicted'] = [LABELS[i] for i in probabilities.argmax(1)]
    subset['confidence'] = probabilities.max(1)
    subset.drop(columns='path').to_csv(OUTPUT_DIR / f'{name}_predictions.csv', index=False)
    wrong = np.flatnonzero((probabilities.argmax(1) != loader.classes) & (probabilities.max(1) >= .95))
    print(f'{name}: {len(wrong)} wrong predictions at >=95% confidence')
    for i in wrong[:8]:
        print(subset.iloc[i][['relative_path', 'label_name', 'predicted', 'confidence']].to_dict())
ax.set(xlabel='Mean predicted confidence', ylabel='Observed accuracy', xlim=(0,1), ylim=(0,1))
ax.legend(); plt.tight_layout(); plt.show()
(OUTPUT_DIR / 'evaluation.json').write_text(json.dumps(results, indent=2))


## Export and verify the Android model
The exported file uses the filename already loaded by BananaQ. Input is float32 RGB
`[1, 224, 224, 3]`, scaled to `[0, 1]`; outputs follow `LABELS` exactly.
Conversion permits only built-in TensorFlow Lite operators. A conversion failure must be resolved
before deployment; this notebook does not enable Select TF operators.

After training and verification, download `BananaQ_CNN_Final.tflite` from the output folder and replace
`app/src/main/assets/diseases/BananaQ_CNN_Final.tflite` in Android Studio. Rebuild and test on a device.
The filename is retained for the app loader even though the architecture is now MobileNetV2.
Verification checks numerical parity, not diagnostic correctness. Review the printed sample true/predicted labels. Pillow and Android decoding/resizing can differ at pixel boundaries; use the saved sample RGB tensor as a device-side golden input, then check actual camera/gallery images on-device.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_bytes = converter.convert()
interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]
assert input_info['shape'].tolist() == [1, *IMAGE_SIZE, 3]
assert output_info['shape'].tolist() == [1, len(LABELS)]
assert input_info['dtype'] == np.float32
assert output_info['dtype'] == np.float32
# Compare both runtimes on several test images before saving for Android.
test_data.reset()
images, sample_labels = test_data[0]
keras_scores = model.predict(images[:8], verbose=0)
tflite_scores = []
for image in images[:8]:
    interpreter.set_tensor(input_info['index'], image[np.newaxis].astype(np.float32))
    interpreter.invoke()
    tflite_scores.append(interpreter.get_tensor(output_info['index'])[0].copy())
tflite_scores = np.asarray(tflite_scores)
np.testing.assert_allclose(tflite_scores, keras_scores, rtol=1e-3, atol=1e-4)
assert np.array_equal(tflite_scores.argmax(axis=1), keras_scores.argmax(axis=1))
TFLITE_PATH = OUTPUT_DIR / 'BananaQ_CNN_Final.tflite'
TFLITE_PATH.write_bytes(tflite_bytes)
(OUTPUT_DIR / 'labels.txt').write_text('\n'.join(LABELS) + '\n')
(OUTPUT_DIR / 'model_info.json').write_text(json.dumps({
    'architecture': 'MobileNetV2', 'labels': LABELS,
    'input_shape': [1, *IMAGE_SIZE, 3], 'input_dtype': 'float32',
    'input_normalization': 'RGB / 255.0; MobileNetV2 normalization embedded in model',
    'resize': 'nearest, stretch to 224x224', 'seed': SEED,
    'class_folders': [p.name for p in class_dirs(next(iter(supplied_splits.values())) if supplied_splits else DATASET_ROOT)],
    'validation_fraction': VALIDATION_FRACTION, 'test_fraction': TEST_FRACTION,
    'supplied_splits': bool(supplied_splits), 'source_groups_provided': GROUP_CSV is not None,
    'class_weights': class_weight, 'run_id': RUN_ID,
    'tensorflow_version': tf.__version__,
}, indent=2))
print('Keras / TFLite verification passed on', len(tflite_scores), 'images.')
print('Android model:', TFLITE_PATH)
print(f'Model size: {len(tflite_bytes) / (1024 * 1024):.2f} MiB')
print('All outputs:', OUTPUT_DIR)

assert np.isfinite(tflite_scores).all()
assert (tflite_scores >= -1e-6).all()
np.testing.assert_allclose(tflite_scores.sum(axis=1), 1, atol=1e-5)
images[0].astype('<f4').tofile(OUTPUT_DIR / 'sample_input_rgb_0_to_1.bin')
(OUTPUT_DIR / 'sample_expected.json').write_text(json.dumps({
    'true_label': LABELS[int(sample_labels[0].argmax())],
    'predicted_label': LABELS[int(tflite_scores[0].argmax())],
    'probabilities': tflite_scores[0].tolist(),
    'tensor_byte_order': 'little-endian float32, NHWC RGB'}, indent=2))
print('Sample true:', LABELS[int(sample_labels[0].argmax())],
      'predicted:', LABELS[int(tflite_scores[0].argmax())])


## Run and deployment checklist
1. Open this notebook in Colab, choose a GPU, and put the ZIP in My Drive. Set `DATASET_ROOT_OVERRIDE` / `GROUP_CSV` only if needed.
2. Choose **Runtime → Run all**. The cell under **Model, callbacks and training** calls `model.fit(...)`; prior cells must run first.
3. Inspect final distributions, learning curves, class supports/F1, confidence errors and reliability plots. High accuracy on related or curated images does not establish field generalization.
4. Only after conversion and parity checks succeed, copy the run folder's `BananaQ_CNN_Final.tflite` into `app/src/main/assets/diseases/`, rebuild, and test with the golden tensor and camera/gallery photos. No Android preprocessing or label change is required.

No training results are supplied with this cleaned notebook. Dataset counts, performance, calibration, conversion success and device latency must be measured by executing it. Environment versions and manifests are saved per run; identical results across different hardware/software are not guaranteed.

Reference: [TensorFlow deterministic execution](https://www.tensorflow.org/api_docs/python/tf/config/experimental/enable_op_determinism) and [TFLite conversion](https://www.tensorflow.org/api_docs/python/tf/lite/TFLiteConverter).
